In [55]:
# --- Plot the Delta shapefile on a folium basemap + (optional) compute monthly ET over the Delta in EE ---

from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium

import ee
from IPython.display import display

# -------------------------
# CONFIG
# -------------------------
delta_shp = Path("../data/regions/Delta_UCB_WGS84/Delta_UCB_WGS84.shp")
assert delta_shp.exists(), f"Not found: {delta_shp.resolve()}"

START = "2000-01-01"  # set to "2012-01-01"
# Use first day of next month so there is ALWAYS at least one full monthly bin
END = (pd.Timestamp.today().normalize() + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")

# Toggle ET computation (folium-only vs folium + EE)
RUN_EE_ET = True

# -------------------------
# GEOMETRY CHOICE
# -------------------------
GEOM = "delta_polygon"  # aggregate over the Delta_UCB_WGS84 shapefile polygon (most precise)
# GEOM = "poly_bbox"       # aggregate over the tight bounding box of the shapefile polygon
# GEOM = "okavango_bbox"  # aggregate over a manually specified rectangular bounding box (faster, larger)


poly_bbox = (21.7913, -21.18, 25., -18.2694)  # (min_lon, min_lat, max_lon, max_lat)

# Output directory for figures / CSVs (one subfolder per geometry)
fig_dir = Path("../figures/ET comparison") / GEOM
fig_dir.mkdir(parents=True, exist_ok=True)
print(f"Geometry: {GEOM}   |   Output dir: {fig_dir.resolve()}")

# -------------------------
# 1) Load shapefile (WGS84) + folium basemap
# -------------------------
gdf = gpd.read_file(delta_shp).to_crs(epsg=4326)
if gdf.empty:
    raise ValueError(f"Shapefile has no features: {delta_shp}")

delta_union = gdf.unary_union
centroid = delta_union.centroid

# Tight bounding box of the delta polygon
minx, miny, maxx, maxy = delta_union.bounds
okavango_bbox = (minx, miny, maxx, maxy)
print(f"Polygon tight bbox: lon {minx:.4f}–{maxx:.4f}°E, lat {miny:.4f}–{maxy:.4f}°")

m = folium.Map(location=[centroid.y, centroid.x], zoom_start=8, tiles=None)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer("CartoDB positron", name="Carto Positron").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri World Imagery",
    overlay=False,
    control=True,
).add_to(m)

folium.GeoJson(
    gdf.__geo_interface__,
    name="Delta polygon",
    style_function=lambda feat: {
        "color": "#8c2d04",
        "weight": 3,        
        "fillOpacity": 0.1,
    },
).add_to(m)

# Tight bbox derived from the polygon perimeter
folium.Rectangle(
    bounds=[[poly_bbox[1], poly_bbox[0]], [poly_bbox[3], poly_bbox[2]]],
    name="Polygon tight bbox",
    color="#fdae6b",
    weight=2,
    fill=False,
    tooltip=f"Polygon bbox: lon {poly_bbox[0]:.2f}–{poly_bbox[2]:.2f}°E, lat {poly_bbox[1]:.2f}–{poly_bbox[3]:.2f}°",
).add_to(m)

# Manual okavango bbox
folium.Rectangle(
    bounds=[[okavango_bbox[1], okavango_bbox[0]], [okavango_bbox[3], okavango_bbox[2]]],
    name="okavango_bbox",
    color="#d95f02",
    fill=True,
    tooltip=f"okavango_bbox: lon {okavango_bbox[0]}–{okavango_bbox[2]}°E, lat {okavango_bbox[1]}–{okavango_bbox[3]}°",
).add_to(m)

# Highlight the active geometry
_active_color = {"delta_polygon": "#d95f02", "poly_bbox": "#e7298a", "okavango_bbox": "#e6ab02"}
_active_color = {"delta_polygon": "#8c2d04", "poly_bbox": "#fdae6b", "okavango_bbox": "#d95f02"}
_active_color = {"delta_polygon": "#8c2d04", "poly_bbox": "#fdae6b", "okavango_bbox": "#d95f02"}

folium.LayerControl(collapsed=False).add_to(m)

Geometry: delta_polygon   |   Output dir: /Users/octaviacrompton/Projects/dswe-inman-lyons/figures/ET comparison/delta_polygon
Polygon tight bbox: lon 21.7913–24.0332°E, lat -20.1858–-18.2694°


In [56]:
# --- Save the folium map to figures/ET comparison ---

# 1) Always save the interactive HTML version
html_path = fig_dir / "delta_map.html"
m.save(str(html_path))
print(f"Saved interactive map → {html_path.resolve()}")

# 2) Try to export a static PNG screenshot (requires selenium + a webdriver)
png_path = fig_dir / "delta_map.png"
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options as ChromeOptions
    import time, io
    from PIL import Image

    opts = ChromeOptions()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1400,1000")
    driver = webdriver.Chrome(options=opts)
    driver.get(f"file://{html_path.resolve()}")
    time.sleep(3)  # let tiles load
    png_data = driver.get_screenshot_as_png()
    driver.quit()

    Image.open(io.BytesIO(png_data)).save(str(png_path))
    print(f"Saved static PNG  → {png_path.resolve()}")
except Exception as exc:
    print(f"PNG export skipped ({type(exc).__name__}: {exc})")
    print("Install selenium + chromedriver for PNG export.  HTML file is still saved above.")

Saved interactive map → /Users/octaviacrompton/Projects/dswe-inman-lyons/figures/ET comparison/delta_polygon/delta_map.html
Saved static PNG  → /Users/octaviacrompton/Projects/dswe-inman-lyons/figures/ET comparison/delta_polygon/delta_map.png


In [ ]:
# -------------------------
# 2) Earth Engine monthly ET over the delta (chunked to avoid 429 errors)
# -------------------------
# ee.Authenticate()  # run once per environment if needed
ee.Initialize()

# Build EE geometry from the GEOM choice made in cell 1
if GEOM == "okavango_bbox":
    delta_geom = ee.Geometry.Rectangle(
        [okavango_bbox[0], okavango_bbox[1], okavango_bbox[2], okavango_bbox[3]],
        proj="EPSG:4326", geodesic=True
    )
    print(f"Geometry: OKAVANGO_BBOX {okavango_bbox}")
elif GEOM == "poly_bbox":
    delta_geom = ee.Geometry.Rectangle(
        [poly_bbox[0], poly_bbox[1], poly_bbox[2], poly_bbox[3]],
        proj="EPSG:4326", geodesic=True
    )
    print(f"Geometry: POLY_BBOX {poly_bbox}")
else:  # "delta_polygon"
    delta_geom = ee.Geometry(delta_union.__geo_interface__)
    print(f"Geometry: DELTA POLYGON ({delta_shp.name})")

delta_area_m2 = float(delta_geom.area(maxError=1).getInfo())
print(f"Region area: {delta_area_m2 / 1e6:.0f} km²")


def monthly_sequence(start_date: str, end_date: str) -> ee.List:
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    n_months = end.difference(start, "month").toInt()
    return ee.List.sequence(0, n_months.subtract(1))

def make_monthly_et_ic(ic: ee.ImageCollection, to_mm_fn, start_date: str, end_date: str) -> ee.ImageCollection:
    """
    Monthly ImageCollection with band 'et_mm' (mm/month) from an input collection.
    Sets system:index = YYYYMM so toBands has clean names.
    """
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    n_months = end.difference(start, "month").toInt()

    def month_img(m):
        m = ee.Number(m)
        m0 = start.advance(m, "month")
        m1 = m0.advance(1, "month")
        sub = ic.filterDate(m0, m1).map(to_mm_fn)

        mm = ee.Image(ee.Algorithms.If(sub.size().gt(0), sub.sum(), ee.Image.constant(0)))
        return (mm.rename("et_mm")
                  .set({
                      "system:time_start": m0.millis(),
                      "system:index": m0.format("YYYYMM"),   # critical for toBands naming
                      "ym": m0.format("YYYY-MM")
                  }))

    months = ee.List(ee.Algorithms.If(n_months.gt(0), monthly_sequence(start_date, end_date), ee.List([])))
    return ee.ImageCollection.fromImages(months.map(month_img))

def monthly_totals_m3_chunked(monthly_ic: ee.ImageCollection, region: ee.Geometry, scale_m: int,
                              chunk_months: int = 24, tile_scale: int = 8) -> dict:
    """
    Returns dict: { 'YYYYMM_et_m3': total_m3_over_region, ... }
    Uses chunked toBands+reduceRegion to avoid "Too many concurrent aggregations".
    """
    n = int(monthly_ic.size().getInfo())
    if n == 0:
        return {}

    imgs = monthly_ic.toList(n)
    out = {}

    for i in range(0, n, chunk_months):
        sub = ee.ImageCollection.fromImages(imgs.slice(i, min(i + chunk_months, n)))

        def to_m3(img):
            img = ee.Image(img)
            idx = ee.String(img.get("system:index"))
            et_m3 = (img.select("et_mm")
                       .unmask(0)
                       .divide(1000)
                       .multiply(ee.Image.pixelArea())
                       .rename("et_m3"))
            return et_m3.set("system:index", idx)

        bands_img = sub.map(to_m3).toBands()

        d = bands_img.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=region,
            scale=scale_m,
            maxPixels=1e13,
            tileScale=tile_scale
        ).getInfo()

        out.update(d)

    return out

def totals_dict_to_df(totals: dict, dataset_name: str, area_m2: float) -> pd.DataFrame:
    rows = []
    for k, v in totals.items():
        yyyymm = k.split("_")[0]
        date = pd.to_datetime(yyyymm + "01", format="%Y%m%d")
        total_m3 = float(v) if v is not None else 0.0
        rows.append({
            "dataset": dataset_name,
            "date": date,
            "et_km3_total": total_m3 / 1e9,
            "et_mm_mean": (total_m3 / area_m2) * 1000.0
        })
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    return df

# -------------------------
# Converters to mm per native timestep (then monthly sum happens in make_monthly_et_ic)
# -------------------------
def mod16_mm(img):
    # ET band = 8-day sum (kg/m²/8-day), scale 0.1 → mm per 8-day composite
    return img.select("ET").multiply(0.1)

def terraclimate_mm(img):
    # aet band = monthly accumulated mm, scale 0.1 → mm/month
    return img.select("aet").multiply(0.1)

def fldas_mm(img):
    # Evap_tavg = monthly mean rate (kg/m²/s = mm/s); multiply by seconds in month → mm/month
    d0 = ee.Date(img.get("system:time_start"))
    seconds = d0.advance(1, "month").difference(d0, "second")
    return img.select("Evap_tavg").multiply(seconds)

def era5_land_mm(img):
    # total_evaporation_sum = monthly accumulated evaporation in m (negative sign convention)
    # multiply by -1000 to get positive mm/month
    return img.select("total_evaporation_sum").multiply(-1000)

def pml_mm(img):
    # PML_V22a: ET band = daily-average rate (raw * 0.01 = mm/day), 8-day composite
    # multiply by 0.01 (scale) × 8 (days per composite) → mm per 8-day image
    return img.select("ET").multiply(0.01).multiply(8)

def wapor_mm(img):
    # L1-AETI-D = dekadal ET rate (raw * 0.1 = mm/day); multiply by actual days in dekad → mm/dekad
    d0 = ee.Date(img.get("system:time_start"))
    d1_candidate = d0.advance(10, "day")
    month_end = ee.Date.fromYMD(d0.get("year"), d0.get("month"), 1).advance(1, "month")
    d1 = ee.Date(ee.Algorithms.If(d1_candidate.millis().lte(month_end.millis()), d1_candidate, month_end))
    days = d1.difference(d0, "day")
    return img.select("L1-AETI-D").multiply(0.1).multiply(days)

DATASETS = [
    dict(name="MOD16A2GF_v61", id="MODIS/061/MOD16A2GF", to_mm=mod16_mm, scale=500,  start="2000-01-01"),
    dict(name="PML_v2_landET", id="projects/pml_evapotranspiration/PML/OUTPUT/PML_V22a", to_mm=pml_mm, scale=500, start="2000-01-01"),
    dict(name="TerraClimate_aet", id="IDAHO_EPSCOR/TERRACLIMATE", to_mm=terraclimate_mm, scale=4638, start="1958-01-01"),
    dict(name="FLDAS_Evap", id="NASA/FLDAS/NOAH01/C/GL/M/V001", to_mm=fldas_mm, scale=11132, start="1982-01-01"),
    dict(name="ERA5Land_totalET", id="ECMWF/ERA5_LAND/MONTHLY_AGGR", to_mm=era5_land_mm, scale=11132, start="1950-02-01"),
    dict(name="USGS_SSEBop_MODIS_monthly",
         id="projects/earthengine-legacy/assets/projects/usgs-ssebop/modis_et_v5_monthly",
         to_mm=lambda img: img.select("et"),
         scale=1000,
         start="2003-01-01"),
    dict(name="WaPORv3_AETI_dekadal", id="FAO/WAPOR/3/L1_AETI_D", to_mm=wapor_mm, scale=248, start="2018-01-01"),
]

# -------------------------
# Run all datasets
# -------------------------
dfs = []
for ds in DATASETS:
    start = max(pd.to_datetime(START), pd.to_datetime(ds["start"])).strftime("%Y-%m-%d")
    if pd.to_datetime(start) >= pd.to_datetime(END):
        print(f"Skipping {ds['name']}: start >= END ({start} >= {END})")
        continue

    ic = (ee.ImageCollection(ds["id"])
          .filterBounds(delta_geom)
          .filterDate(start, END))

    monthly_ic = make_monthly_et_ic(ic, ds["to_mm"], start, END)
    n_months = int(monthly_ic.size().getInfo())
    print(f"{ds['name']}: monthly bins = {n_months} | start={start} end={END}")

    totals = monthly_totals_m3_chunked(monthly_ic, delta_geom, scale_m=ds["scale"], chunk_months=24, tile_scale=8)
    df_ds = totals_dict_to_df(totals, ds["name"], area_m2=delta_area_m2)
    dfs.append(df_ds)

df_all = pd.concat(dfs, ignore_index=True).sort_values(["dataset", "date"]).reset_index(drop=True)


Geometry: DELTA POLYGON (Delta_UCB_WGS84.shp)
Region area: 23387 km²
MOD16A2GF_v61: monthly bins = 314 | start=2000-01-01 end=2026-04-01
PML_v2_landET: monthly bins = 314 | start=2000-01-01 end=2026-04-01
TerraClimate_aet: monthly bins = 314 | start=2000-01-01 end=2026-04-01
FLDAS_Evap: monthly bins = 314 | start=2000-01-01 end=2026-04-01
ERA5Land_totalET: monthly bins = 314 | start=2000-01-01 end=2026-04-01
USGS_SSEBop_MODIS_monthly: monthly bins = 278 | start=2003-01-01 end=2026-04-01


In [ ]:
# import pandas as pd
import matplotlib.pyplot as plt

# Ensure datetime
df = df_all.copy()
df["date"] = pd.to_datetime(df["date"])

#  restrict to 2012+
# df = df[df["date"] >= "2012-01-01"]

# --- Plot 1: area-mean ET (mm/month) ---
wide_mm = (df.pivot_table(index="date", columns="dataset", values="et_mm_mean", aggfunc="mean")
             .sort_index())

fig, ax = plt.subplots(figsize=(12, 5))
wide_mm.plot(ax=ax, linewidth=1)
ax.set_title("Okavango Delta monthly ET (area-mean)")
ax.set_ylabel("ET (mm / month)")
ax.set_xlabel("")
ax.legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()


In [ ]:
df_all['month'] = df_all.date.dt.month
# --- Plot 1: area-mean ET (mm/month) ---
wide_mm = (df_all.pivot_table(index="month", columns="dataset", values="et_mm_mean", aggfunc="mean")
             .sort_index())

fig, ax = plt.subplots(figsize=(12, 5))
wide_mm.plot(ax=ax, linewidth=1)
ax.set_title("Okavango Delta monthly ET (area-mean)")
ax.set_ylabel("ET (mm / month)")
ax.set_xlabel("")
ax.legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import ee

# -------------------------
# CONFIG
# -------------------------
# delta_shp = Path("../data/regions/Delta_UCB_WGS84/Delta_UCB_WGS84.shp")
# assert delta_shp.exists(), f"Not found: {delta_shp.resolve()}"

START = "2000-01-01"  # or "2012-01-01"
END = (pd.Timestamp.today().normalize() + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")

OUTDIR = Path("../data/chirps")
OUTDIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# EE init
# -------------------------
# ee.Authenticate()  # run once per environment if needed
ee.Initialize()

# -------------------------
# Load delta geometry
# -------------------------
# gdf = gpd.read_file(delta_shp).to_crs(epsg=4326)
# delta_union = gdf.unary_union
# delta_geom = ee.Geometry(delta_union.__geo_interface__)
# delta_area_m2 = float(delta_geom.area(maxError=1).getInfo())

# -------------------------
# Helpers
# -------------------------
def monthly_sequence(start_date: str, end_date: str) -> ee.List:
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    n_months = end.difference(start, "month").toInt()
    return ee.List.sequence(0, n_months.subtract(1))

def make_monthly_from_daily(daily_ic: ee.ImageCollection, band: str, start_date: str, end_date: str) -> ee.ImageCollection:
    """
    Sum daily precipitation into monthly totals (mm/month).
    """
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    n_months = end.difference(start, "month").toInt()

    def month_img(m):
        m = ee.Number(m)
        m0 = start.advance(m, "month")
        m1 = m0.advance(1, "month")
        sub = daily_ic.filterDate(m0, m1).select(band)

        mm = ee.Image(ee.Algorithms.If(sub.size().gt(0), sub.sum(), ee.Image.constant(0)))
        return (mm.rename("ppt_mm")
                  .set({
                      "system:time_start": m0.millis(),
                      "system:index": m0.format("YYYYMM"),
                      "ym": m0.format("YYYY-MM")
                  }))

    months = ee.List(ee.Algorithms.If(n_months.gt(0), monthly_sequence(start_date, end_date), ee.List([])))
    return ee.ImageCollection.fromImages(months.map(month_img))

def collection_depth_to_totals_dict(ic: ee.ImageCollection,
                                    band: str,
                                    date_fmt: str,
                                    region: ee.Geometry,
                                    scale_m: int,
                                    chunk_n: int,
                                    tile_scale: int = 8) -> dict:
    """
    Reduce an ImageCollection of depth images (mm per image) to a dict of total m3 over region.
    Returns keys like 'YYYYMM_ppt_m3' or 'YYYYMMdd_ppt_m3'.
    """
    n = int(ic.size().getInfo())
    if n == 0:
        return {}

    imgs = ic.toList(n)
    out = {}

    def prep(img):
        img = ee.Image(img)
        d = ee.Date(img.get("system:time_start"))
        idx = d.format(date_fmt)
        m3 = (img.select(band)
                .unmask(0)
                .divide(1000)  # mm -> m
                .multiply(ee.Image.pixelArea())
                .rename("ppt_m3"))
        return m3.set("system:index", idx)

    for i in range(0, n, chunk_n):
        sub = ee.ImageCollection.fromImages(imgs.slice(i, min(i + chunk_n, n))).map(prep)
        bands_img = sub.toBands()  # band names become YYYYMM_ppt_m3 etc.

        dct = bands_img.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=region,
            scale=scale_m,
            maxPixels=1e13,
            tileScale=tile_scale
        ).getInfo()

        out.update(dct)

    return out

def totals_dict_to_df(totals: dict, date_parse: str, dataset_name: str, area_m2: float) -> pd.DataFrame:
    """
    totals keys like 'YYYYMM_ppt_m3' -> date, ppt_km3_total, ppt_mm_mean
    """
    rows = []
    for k, v in totals.items():
        ymd = k.split("_")[0]
        date = pd.to_datetime(ymd + ("01" if date_parse == "%Y%m" else ""), format=date_parse + ("" if date_parse != "%Y%m" else "%d"))
        total_m3 = float(v) if v is not None else 0.0
        rows.append({
            "dataset": dataset_name,
            "date": date,
            "ppt_km3_total": total_m3 / 1e9,
            "ppt_mm_mean": (total_m3 / area_m2) * 1000.0
        })
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    return df

# -------------------------
# 1) CHIRPS MONTHLY over delta
# -------------------------
# Option A (preferred if available): monthly collection directly
#   monthly_id = "UCSB-CHG/CHIRPS/MONTHLY"
# Option B (always works): make monthly from CHIRPS daily
daily_id = "UCSB-CHG/CHIRPS/DAILY"
band_daily = "precipitation"

chirps_daily = (ee.ImageCollection(daily_id)
                .filterBounds(delta_geom)
                .filterDate(START, END))

chirps_monthly = make_monthly_from_daily(chirps_daily, band_daily, START, END)  # mm/month in band 'ppt_mm'

tot_m = collection_depth_to_totals_dict(
    ic=chirps_monthly,
    band="ppt_mm",
    date_fmt="YYYYMM",
    region=delta_geom,
    scale_m=5500,     # CHIRPS ~0.05° (~5km)
    chunk_n=120,      # months per request (10 years)
    tile_scale=8
)

df_chirps_monthly = totals_dict_to_df(tot_m, date_parse="%Y%m", dataset_name="CHIRPS_monthly", area_m2=delta_area_m2)
out_monthly = OUTDIR / "chirps_delta_monthly.csv"
df_chirps_monthly.to_csv(out_monthly, index=False)
print("Saved:", out_monthly, "| rows:", len(df_chirps_monthly))

# -------------------------
# 2)  CHIRPS DAILY over delta
# -------------------------
# Uncomment for daily series; consider START="2012-01-01" to keep it lighter.

# def add_index(img):
#     d = ee.Date(img.get("system:time_start"))
#     return ee.Image(img).set("system:index", d.format("YYYYMMdd"))
#
# chirps_daily_idx = chirps_daily.map(add_index)
#
# tot_d = collection_depth_to_totals_dict(
#     ic=chirps_daily_idx.select(band_daily),
#     band=band_daily,
#     date_fmt="YYYYMMdd",
#     region=delta_geom,
#     scale_m=5500,
#     chunk_n=180,     # ~6 months per request (keeps bands reasonable)
#     tile_scale=8
# )
#
# df_chirps_daily = totals_dict_to_df(tot_d, date_parse="%Y%m%d", dataset_name="CHIRPS_daily", area_m2=delta_area_m2)
# out_daily = OUTDIR / "chirps_delta_daily.csv"
# df_chirps_daily.to_csv(out_daily, index=False)
# print("Saved:", out_daily, "| rows:", len(df_chirps_daily))


In [ ]:

# --- Plot 1: area-mean ET (mm/month) ---
wide_mm = (df.pivot_table(index="date", columns="dataset", values="et_mm_mean", aggfunc="mean")
             .sort_index())


fig, ax = plt.subplots(figsize=(12, 5))
wide_mm.plot(ax=ax, linewidth=1)
ax.set_title("Okavango Delta monthly ET (area-mean)")
ax.set_ylabel("ET (mm / month)")
ax.set_xlabel("")
ax.legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
# plt.tight_layout()
# plt.show()
# df_chirps_monthly.set_index('date')['ppt_mm_mean'].plot(lw = 2, ls = '--', c = 'g')

In [ ]:

# --- Drop WaPORv3 and TerraClimate from all downstream analyses ---
_drop_datasets = ["WaPORv3_AETI_dekadal", "TerraClimate_aet"]
wide_mm = wide_mm.drop(columns=[c for c in _drop_datasets if c in wide_mm.columns])
df = df[~df["dataset"].isin(_drop_datasets)].copy()
df_all = df_all[~df_all["dataset"].isin(_drop_datasets)].copy()

# --- Rank datasets by mean ET, split into highest-4 and lowest-4 ---
mean_et = wide_mm.mean()
top4 = mean_et.nlargest(3).index.tolist()
bot4 = mean_et.nsmallest(3).index.tolist()

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

wide_mm[top4].plot(ax=axes[0], linewidth=1)
axes[0].set_title("3 highest-ET models (by mean mm/month)")
axes[0].set_ylabel("ET (mm / month)")
axes[0].set_xlabel("")
axes[0].legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

wide_mm[bot4].plot(ax=axes[1], linewidth=1)
axes[1].set_title("3 lowest-ET models (by mean mm/month)")
axes[1].set_ylabel("ET (mm / month)")
axes[1].set_xlabel("")
axes[1].legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

plt.tight_layout()
plt.show()

print("Top 3 (highest mean ET):", top4)
print("Bot 3 (lowest mean ET): ", bot4)


In [ ]:
df_chirps_monthly.head()

In [ ]:
chirps_mm = df_chirps_monthly.set_index('date')['ppt_mm_mean']

chirps_km3 = df_chirps_monthly.set_index('date')['ppt_km3_total']

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharey = True)
ax = axes[0]
wide_mm.plot(ax=ax, linewidth=1)
ax.set_title("Okavango Delta monthly ET (area-mean)")
ax.set_ylabel("ET (mm / month)")
ax.set_xlabel("")
ax.legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

chirps_mm.plot(lw = 1, ls = '--', c = 'g')

wide_mm.median(1).plot(ls = '-', c = 'C0')
(chirps_mm - wide_mm.median(1)).plot(c = 'C1')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

start = "2000-01-01"

# --- Slice to common window
wide_mm_2012 = wide_mm.loc[pd.to_datetime(start):]
et_med = wide_mm_2012.median(axis=1)

chirps_mm = (df_chirps_monthly
             .assign(date=pd.to_datetime(df_chirps_monthly["date"]))
             .set_index("date")["ppt_mm_mean"]
             .loc[pd.to_datetime(start):])

# Align indexes before differencing
chirps_mm_aligned, et_med_aligned = chirps_mm.align(et_med, join="inner")
p_minus_et = chirps_mm_aligned - et_med_aligned

# --- Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)  # sharey=False is usually better here
ax0, ax1 = axes

# Panel 1: ET datasets + ET median + CHIRPS precip
wide_mm_2012.plot(ax=ax0, linewidth=1)
et_med.plot(ax=ax1, lw=1, c="k", label="ET median (across products)")
chirps_mm.plot(ax=ax1, lw=1, ls="--", c="#7fcdbb", label="CHIRPS precipitation")

ax0.set_title("Okavango Delta monthly ET and precipitation (area-mean)")
ax0.set_ylabel("mm / month")
ax0.set_xlabel("")
ax0.legend(title="Series", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

# Panel 2: P - ET
p_minus_et.plot(ax=ax1, lw=1, c="#225ea8", label="P − ET (CHIRPS − ET median)")
ax1.axhline(0, lw=1, ls=":", c="0.4")

ax1.set_title("Monthly water-balance proxy")
ax1.set_ylabel("mm / month")
ax1.set_xlabel("Date")
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:

wide_mm_2012.head()


In [ ]:
import pandas as pd
import numpy as np

ssebop = "USGS_SSEBop_MODIS_monthly"
tiny = 1e-6  # km3 or mm threshold for "effectively zero"

m = df_all["dataset"].eq(ssebop)

# Find last month with a non-trivial value (before we blank zeros)
last_valid = df_all.loc[m & (df_all["et_km3_total"] > tiny), "date"].max()
print("SSEBop last nonzero month:", last_valid)

# 1) Convert zeros/tiny values to NaN (missing)
df_all.loc[m & (df_all["et_km3_total"] <= tiny), "et_km3_total"] = np.nan
df_all.loc[m & (df_all["et_mm_mean"]   <= tiny), "et_mm_mean"]   = np.nan

# 2) Optional: if it truly stops after last_valid, blank everything after that month too
if pd.notna(last_valid):
    df_all.loc[m & (df_all["date"] > last_valid), ["et_km3_total", "et_mm_mean"]] = np.nan
    
    
# Ensure datetime
df_et = df_all.copy()
df_et["date"] = pd.to_datetime(df_et["date"])

df_p = df_chirps_monthly.copy()
df_p["date"] = pd.to_datetime(df_p["date"])

# 1) Pivot ET wide (one column per dataset)
et_mm_wide  = df_et.pivot_table(index="date", columns="dataset", values="et_mm_mean",  aggfunc="median")
et_km3_wide = df_et.pivot_table(index="date", columns="dataset", values="et_km3_total", aggfunc="median")

# 2) Pivot precip wide (in case you later add multiple precip products)
p_mm_wide  = df_p.pivot_table(index="date", columns="dataset", values="ppt_mm_mean",  aggfunc="median")
p_km3_wide = df_p.pivot_table(index="date", columns="dataset", values="ppt_km3_total", aggfunc="median")

# 3) Flatten column names and join
et_mm_wide.columns  = [f"ETmm_{c}" for c in et_mm_wide.columns]
et_km3_wide.columns = [f"ETkm3_{c}" for c in et_km3_wide.columns]
p_mm_wide.columns   = [f"Pmm_{c}" for c in p_mm_wide.columns]
p_km3_wide.columns  = [f"Pkm3_{c}" for c in p_km3_wide.columns]

df_combined = pd.concat([et_mm_wide, et_km3_wide, p_mm_wide, p_km3_wide], axis=1).sort_index()

# Optional: keep only overlap period across all columns
# df_combined = df_combined.dropna(how="any")

df_combined.reset_index(inplace=True)
df_combined.rename(columns={"index": "date"}, inplace=True)


In [ ]:
# --- GRACE (JPL RL06.3 mascon CRI) monthly time series over the Okavango Delta polygon ---
# Produces:
#   df_grace: date, grace_km3_total, grace_cm_mean
# And (optionally) merges into your existing df_combo on date.

from pathlib import Path
import re
import geopandas as gpd
import pandas as pd
import ee

# -------------------------
# CONFIG
# -------------------------
# delta_shp = Path("../data/regions/Delta_UCB_WGS84/Delta_UCB_WGS84.shp")
# assert delta_shp.exists(), f"Not found: {delta_shp.resolve()}"

START = "2002-04-01"  # GRACE starts ~2002
END   = (pd.Timestamp.today().normalize() + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")

GRACE_ID = "NASA/GRACE/MASS_GRIDS_V04/MASCON_CRI"   # CRI-filtered JPL mascon (monthly)
BAND = "lwe_thickness"                             # cm equivalent water thickness anomaly

# # -------------------------
# # EE init + geometry
# # -------------------------
# # ee.Authenticate()  # run once per environment if needed
# ee.Initialize()

# gdf = gpd.read_file(delta_shp).to_crs(epsg=4326)
# delta_union = gdf.unary_union
# delta_geom = ee.Geometry(delta_union.__geo_interface__)
# delta_area_m2 = float(delta_geom.area(maxError=1).getInfo())

# -------------------------
# Helpers
# -------------------------
def grace_totals_m3_chunked(grace_ic: ee.ImageCollection,
                            region: ee.Geometry,
                            scale_m: int = 55660,
                            chunk_n: int = 60,
                            tile_scale: int = 8) -> dict:
    """
    Reduce GRACE IC to dict of total m^3 anomaly over region for each monthly image.
    Keys look like 'YYYYMM_tws_m3' (sometimes prefixed like '0_YYYYMM_tws_m3').
    """
    n = int(grace_ic.size().getInfo())
    if n == 0:
        return {}

    imgs = grace_ic.toList(n)
    out = {}

    def prep(img):
        img = ee.Image(img)
        d = ee.Date(img.get("system:time_start"))
        idx = d.format("YYYYMM")

        # lwe_thickness is cm of equivalent water thickness anomaly
        tws_m3 = (img.select(BAND)
                    .unmask(0)
                    .divide(100.0)              # cm -> m
                    .multiply(ee.Image.pixelArea())
                    .rename("tws_m3"))

        return tws_m3.set("system:index", idx)

    for i in range(0, n, chunk_n):
        sub = ee.ImageCollection.fromImages(imgs.slice(i, min(i + chunk_n, n))).map(prep)
        bands_img = sub.toBands()

        dct = bands_img.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=region,
            scale=scale_m,
            maxPixels=1e13,
            tileScale=tile_scale
        ).getInfo()

        out.update(dct)

    return out

def totals_dict_to_df(totals: dict, area_m2: float) -> pd.DataFrame:
    """
    totals keys may contain YYYYMM or YYYYMMDD anywhere (and may have a numeric prefix).
    Returns monthly time series:
      grace_km3_total: km^3 anomaly over region
      grace_cm_mean: area-mean cm anomaly over region
    """
    rows = []
    for k, v in totals.items():
        m = re.search(r"(\d{8}|\d{6})", k)
        if m is None:
            continue
        code = m.group(1)

        if len(code) == 6:      # YYYYMM
            date = pd.to_datetime(code + "01", format="%Y%m%d")
        else:                   # YYYYMMDD
            date = pd.to_datetime(code, format="%Y%m%d")

        total_m3 = float(v) if v is not None else 0.0
        mean_m = total_m3 / area_m2

        rows.append({
            "date": date,
            "grace_km3_total": total_m3 / 1e9,
            "grace_cm_mean": mean_m * 100.0
        })

    return (pd.DataFrame(rows)
              .sort_values("date")
              .reset_index(drop=True))

# -------------------------
# Build GRACE IC + compute
# -------------------------
grace_ic = (ee.ImageCollection(GRACE_ID)
            .filterBounds(delta_geom)
            .filterDate(START, END)
            .select(BAND))

print("GRACE images:", grace_ic.size().getInfo())

totals = grace_totals_m3_chunked(
    grace_ic=grace_ic,
    region=delta_geom,
    scale_m=55660,   # ~55 km mascon grid
    chunk_n=60,      # ~5 years per request; reduce if you get 429
    tile_scale=8     # increase to 16 if you get 429
)

# Optional: inspect a few keys if debugging
print("Example keys:", list(totals.keys())[:8])

df_grace = totals_dict_to_df(totals, area_m2=delta_area_m2)



In [ ]:
# fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharey = False)
# ax = axes[0]
# (chirps_mm - wide_mm.median(1)).plot(c = 'C1', ax = ax, marker = '.', linestyle = ':')
# ax = axes[1]
# df_grace.set_index('date')['grace_cm_mean'].plot(marker = '.', linestyle = ':')

In [ ]:
ET_km3 = (df.pivot_table(index="date", columns="dataset", values="et_km3_total", aggfunc="mean")
             .sort_index())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, sharey=False)

# Panel 1: P - ET
ax0 = axes[0]
(chirps_km3 - ET_km3.median(1)).plot(
    ax=ax0, c="#225ea8", marker=".", linestyle=":", label="P − ET (CHIRPS − ET median)"
)
ax0.axhline(0, lw=1, ls=":", c="0.4")
ax0.set_title("Monthly water-balance proxy vs GRACE anomaly (Delta area-mean)")
ax0.set_ylabel("mm / month")
ax0.set_xlabel("")
ax0.legend(loc="upper left")

# Panel 2: GRACE
ax1 = axes[1]
(df_grace.set_index("date")["grace_km3_total"]).plot(
    ax=ax1, marker=".", linestyle=":", c="#1d91c0", label="GRACE/GRACE-FO: equivalent water thickness anomaly"
)
ax1.axhline(0, lw=1, ls=":", c="0.4")
ax1.set_ylabel("km3 (anomaly)")
ax1.set_xlabel("Date")
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, sharey=False)

# Panel 1: P - ET
ax0 = axes[0]
(chirps_mm - wide_mm.median(1)).plot(
    ax=ax0, c="#225ea8", marker=".", linestyle=":", label="P − ET (CHIRPS − ET median)"
)
ax0.axhline(0, lw=1, ls=":", c="0.4")
ax0.set_title("Monthly water-balance proxy vs GRACE anomaly (Delta area-mean)")
ax0.set_ylabel("mm / month")
ax0.set_xlabel("")
ax0.legend(loc="upper left")

# Panel 2: GRACE
ax1 = axes[1]
(df_grace.set_index("date")["grace_cm_mean"]).plot(
    ax=ax1, marker=".", linestyle=":", c="#1d91c0", label="GRACE/GRACE-FO: equivalent water thickness anomaly"
)
ax1.axhline(0, lw=1, ls=":", c="0.4")
ax1.set_ylabel("cm (anomaly)")
ax1.set_xlabel("Date")
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
df_grace.head(10)

In [ ]:
# Mohembo/Mtaembo (GRDC 1357100) — read pre-built CSVs
# (Canonical source: "Mohembo time series.ipynb" writes these files)
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DAILY_CSV   = Path("../data/mohembo_1357100_Q_daily.csv")
MONTHLY_CSV = Path("../data/mohembo_1357100_Q_monthly_mean.csv")

ts = pd.read_csv(DAILY_CSV, parse_dates=["date"], index_col="date")
monthly = pd.read_csv(MONTHLY_CSV, parse_dates=["month"], index_col="month")

print(f"Daily rows: {len(ts):,} | Range: {ts.index.min().date()} → {ts.index.max().date()}")
print(f"Monthly rows: {len(monthly):,}")

# Plot (daily + 30-day mean)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.index, ts["Q_m3s"], linewidth=0.6, label="Daily Q")
q_30d = ts["Q_m3s"].rolling("30D", min_periods=10).mean()
ax.plot(q_30d.index, q_30d, linewidth=1.2, label="30-day mean")
ax.set_title("MOHEMBO/MTAEMBO (GRDC 1357100) — Mean daily discharge")
ax.set_xlabel("Date")
ax.set_ylabel(r"Discharge $Q$ (m$^3$/s)")
ax.grid(True, linewidth=0.2)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

# ----------------------------
# 0) Inputs / paths
# ----------------------------
# delta_shp = Path("../data/regions/Delta_UCB_WGS84/Delta_UCB_WGS84.shp")
mohembo_monthly_csv = Path("../data/mohembo_1357100_Q_monthly_mean.csv")  # from your notebook cell

# Delta area (m²) using equal-area projection
# delta_area_m2 = (
#     gpd.read_file(delta_shp)
#       .to_crs(epsg=6933)  # world equal-area
#       .geometry.area
#       .sum()
# )


def to_month_start(x):
    """Normalize datetimes to month-start for clean monthly merges."""
    x = pd.to_datetime(x)
    if isinstance(x, (pd.DatetimeIndex, pd.PeriodIndex)):
        return x.to_period("M").to_timestamp(how="start")
    return x.dt.to_period("M").dt.to_timestamp(how="start")

# ----------------------------
# 1) Mohembo inflow (monthly mean Q) -> monthly volumes (km³) + equivalent depth (mm)
# ----------------------------
q = pd.read_csv(mohembo_monthly_csv)

# file saved with index_label="month" OR may have a first column that is the month index
month_col = "month" if "month" in q.columns else q.columns[0]
q = q.rename(columns={month_col: "date"})
q["date"] = to_month_start(q["date"])

# column name from your notebook
q_col = "Q_m3s_monthly_mean"
if q_col not in q.columns:
    # fallback: pick the first numeric column
    q_col = q.select_dtypes(include="number").columns[0]

q = q.rename(columns={q_col: "Qin_m3s"})

# seconds in each month
seconds_in_month = (q["date"].dt.days_in_month * 24 * 3600).astype(float)

q["Qin_km3"] = q["Qin_m3s"] * seconds_in_month / 1e9
q["Qin_mm"]  = q["Qin_m3s"] * seconds_in_month / delta_area_m2 * 1000.0
q_mohembo = q[["date", "Qin_m3s", "Qin_km3", "Qin_mm"]].sort_values("date")

# ----------------------------
# 2) Build ONE master dataframe: ET + CHIRPS + GRACE + Mohembo
# ----------------------------
# Ensure datetime + normalize to month-start
df_et = df_all.copy()
df_et["date"] = to_month_start(df_et["date"])

df_p = df_chirps_monthly.copy()
df_p["date"] = to_month_start(df_p["date"])

df_g = df_grace.copy()
df_g["date"] = to_month_start(df_g["date"])

# If GRACE has multiple entries per month, collapse
df_g = df_g.groupby("date", as_index=False).mean(numeric_only=True)

# ET wide
et_mm  = df_et.pivot_table(index="date", columns="dataset", values="et_mm_mean",  aggfunc="mean").add_prefix("ETmm_")
et_km3 = df_et.pivot_table(index="date", columns="dataset", values="et_km3_total", aggfunc="mean").add_prefix("ETkm3_")

# Precip wide
p_mm  = df_p.pivot_table(index="date", columns="dataset", values="ppt_mm_mean",  aggfunc="mean").add_prefix("Pmm_")
p_km3 = df_p.pivot_table(index="date", columns="dataset", values="ppt_km3_total", aggfunc="mean").add_prefix("Pkm3_")

# GRACE (monthly anomaly over delta) -> indexed by month-start
grace = (df_g.set_index("date")[["grace_cm_mean", "grace_km3_total"]]
           .rename(columns={"grace_cm_mean": "GRACEcm_mean",
                            "grace_km3_total": "GRACEkm3_total"}))

# Combine (outer join on date index)
df_master = pd.concat([et_mm, et_km3, p_mm, p_km3, grace], axis=1).sort_index()

# ET medians (across products)
etmm_cols  = [c for c in df_master.columns if c.startswith("ETmm_")]
etkm3_cols = [c for c in df_master.columns if c.startswith("ETkm3_")]
df_master["ETmm_median"]  = df_master[etmm_cols].median(axis=1, skipna=True)
df_master["ETkm3_median"] = df_master[etkm3_cols].median(axis=1, skipna=True)

# Choose CHIRPS precip columns (first match containing CHIRPS; else first precip col)
pmm_cols  = [c for c in df_master.columns if c.startswith("Pmm_")]
pkm3_cols = [c for c in df_master.columns if c.startswith("Pkm3_")]
pmm_col  = next((c for c in pmm_cols  if "CHIRPS" in c.upper()),  (pmm_cols[0] if pmm_cols else None))
pkm3_col = next((c for c in pkm3_cols if "CHIRPS" in c.upper()), (pkm3_cols[0] if pkm3_cols else None))
df_master["Pmm"]  = df_master[pmm_col]  if pmm_col  else np.nan
df_master["Pkm3"] = df_master[pkm3_col] if pkm3_col else np.nan

# Storage change from GRACE (diff of anomaly) — safe only where GRACE exists
df_master["dS_km3"] = df_master["GRACEkm3_total"].diff()
df_master["dS_cm"]  = df_master["GRACEcm_mean"].diff()

# Mask GRACE diffs across big gaps (missing months)
gap_days = df_master.index.to_series().diff().dt.days
df_master.loc[gap_days > 45, ["dS_km3", "dS_cm"]] = np.nan

# Add Mohembo inflow (month-start alignment)
df_master = df_master.merge(q_mohembo.set_index("date"), left_index=True, right_index=True, how="left")

df_master

In [ ]:

# ----------------------------
# 3) Mass-balance helper terms (control volume = Delta polygon)
# ----------------------------
# (Qout + G) implied by: dS = Qin + P - ET - (Qout + G)
df_master["Qout_plus_G_km3"] = df_master["Qin_km3"] + df_master["Pkm3"] - df_master["ETkm3_median"] - df_master["dS_km3"]
df_master["Qout_plus_G_mm"]  = df_master["Qin_mm"]  + df_master["Pmm"]  - df_master["ETmm_median"]  - (df_master["dS_cm"] * 10.0)

# Back to a normal column for plotting
df_master = df_master.reset_index().rename(columns={"index": "date"})

# Quick check plot
import matplotlib.dates as mdates
plt.figure(figsize = (12, 4))
ax = df_master.set_index("date")["GRACEkm3_total"].plot(title="GRACE TWS anomaly over Delta (km³)")
ax.set_ylabel("km³ (anomaly)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


In [ ]:

import numpy as np
import pandas as pd

def to_month_start(x):
    """Normalize datetimes to month-start (avoids GRACE vs ET/CHIRPS merge mismatch)."""
    x = pd.to_datetime(x)
    return x.dt.to_period("M").dt.to_timestamp(how="start")

# --- normalize all dates to month-start (CRITICAL) ---
df_all = df_all.copy()
df_chirps_monthly = df_chirps_monthly.copy()
df_grace = df_grace.copy()
q_mohembo = q_mohembo.copy()

df_all["date"] = to_month_start(df_all["date"])
df_chirps_monthly["date"] = to_month_start(df_chirps_monthly["date"])
df_grace["date"] = to_month_start(df_grace["date"])
q_mohembo["date"] = to_month_start(q_mohembo["date"])

# If any duplicates within a month, collapse
df_all = df_all.groupby(["date", "dataset"], as_index=False).mean(numeric_only=True)
df_chirps_monthly = df_chirps_monthly.groupby(["date", "dataset"], as_index=False).mean(numeric_only=True)
df_grace = df_grace.groupby("date", as_index=False).mean(numeric_only=True)
q_mohembo = q_mohembo.groupby("date", as_index=False).mean(numeric_only=True)

# -----------------------
# 1) ET ensemble median
# -----------------------
et_mm_wide  = df_all.pivot_table(index="date", columns="dataset", values="et_mm_mean",  aggfunc="median").sort_index()
et_km3_wide = df_all.pivot_table(index="date", columns="dataset", values="et_km3_total", aggfunc="median").sort_index()

ETmm_med  = et_mm_wide.median(axis=1, skipna=True).rename("ETmm")
ETkm3_med = et_km3_wide.median(axis=1, skipna=True).rename("ETkm3")

# -----------------------
# 2) CHIRPS precip
# -----------------------
p_mm_wide  = df_chirps_monthly.pivot_table(index="date", columns="dataset", values="ppt_mm_mean",  aggfunc="median").sort_index()
p_km3_wide = df_chirps_monthly.pivot_table(index="date", columns="dataset", values="ppt_km3_total", aggfunc="median").sort_index()

if p_mm_wide.shape[1] == 0 or p_km3_wide.shape[1] == 0:
    raise ValueError("No precip columns found in df_chirps_monthly after pivot. Check 'dataset' values/bands.")

pmm_col  = next((c for c in p_mm_wide.columns  if "CHIRPS" in str(c).upper()),  p_mm_wide.columns[0])
pkm3_col = next((c for c in p_km3_wide.columns if "CHIRPS" in str(c).upper()), p_km3_wide.columns[0])

Pmm  = p_mm_wide[pmm_col].rename("Pmm")
Pkm3 = p_km3_wide[pkm3_col].rename("Pkm3")

# -----------------------
# 3) GRACE storage anomaly + monthly change (with gap checks)
# -----------------------
gr = (df_grace.set_index("date")[["grace_cm_mean", "grace_km3_total"]]
      .sort_index()
      .rename(columns={"grace_cm_mean": "GRACEcm", "grace_km3_total": "GRACEkm3"}))

# sanity: GRACE should now be month-start
bad_grace_dates = gr.index[gr.index.day != 1]
if len(bad_grace_dates) > 0:
    raise ValueError(f"GRACE dates not normalized to month-start (day!=1). Example: {bad_grace_dates[:5].tolist()}")

dS_km3 = gr["GRACEkm3"].diff().rename("dS_km3")
dS_cm  = gr["GRACEcm"].diff().rename("dS_cm")

gap_days = gr.index.to_series().diff().dt.days
# flag likely missing-month gaps
if (gap_days > 45).any():
    n_gaps = int((gap_days > 45).sum())
    print(f"Warning: {n_gaps} GRACE gaps >45 days; setting dS to NaN across those steps.")

dS_km3[gap_days > 45] = np.nan
dS_cm[gap_days > 45]  = np.nan

# -----------------------
# 4) Qin (Mohembo) + mass-balance terms
# -----------------------
qin = (q_mohembo.set_index("date")[["Qin_m3s", "Qin_km3", "Qin_mm"]]
       .sort_index())

# sanity: Qin should be month-start
bad_qin_dates = qin.index[qin.index.day != 1]
if len(bad_qin_dates) > 0:
    raise ValueError(f"Qin dates not normalized to month-start (day!=1). Example: {bad_qin_dates[:5].tolist()}")

df_balance = pd.concat([Pmm, Pkm3, ETmm_med, ETkm3_med, gr, dS_km3, dS_cm, qin], axis=1).sort_index()

# -----------------------
# 5) Error / closure checks
# -----------------------
# implied (Qout + net groundwater export):
df_balance["Qout_plus_G_km3"] = df_balance["Qin_km3"] + df_balance["Pkm3"] - df_balance["ETkm3"] - df_balance["dS_km3"]
df_balance["Qout_plus_G_mm"]  = df_balance["Qin_mm"]  + df_balance["Pmm"]  - df_balance["ETmm"]  - (df_balance["dS_cm"] * 10.0)

# convenience: months where you can actually compute closure (needs Qin, P, ET, and dS)
df_balance["has_closure"] = df_balance[["Qin_km3", "Pkm3", "ETkm3", "dS_km3"]].notna().all(axis=1)

# a residual definition (should be ~0 if you later include Qout and G separately)
# residual = dS - (Qin + P - ET - (Qout+G)) ; since Qout+G is implied here, residual is identically 0 where defined.
# But we can check for impossible values:
bad = df_balance.loc[df_balance["has_closure"] & (df_balance["Qout_plus_G_km3"] < 0)]
if len(bad) > 0:
    print(f"Note: {len(bad)} months have negative implied (Qout+G). "
          "Could be storage/ET/P bias, timing mismatch, or true net inflow > outflow+G not holding for your control volume.")

df_balance.reset_index().rename(columns={"index": "date"})

# Add this at the very end (after df_balance is created)

df_balance = df_balance.loc[df_balance.index.year > 2000].copy()
df_balance.reset_index().rename(columns={"index": "date"})


In [ ]:
# -------------------------
# Save data outputs for this geometry
# -------------------------
# All CSVs land in fig_dir so each geometry has its own copy for later comparison

# ET ensemble
df_all.to_csv(fig_dir / "et_monthly.csv", index=False)

# CHIRPS
df_chirps_monthly.to_csv(fig_dir / "chirps_monthly.csv", index=False)

# GRACE
df_grace.to_csv(fig_dir / "grace_monthly.csv", index=False)

# Mohembo discharge
q_mohembo.to_csv(fig_dir / "mohembo_monthly.csv", index=False)

# Full mass-balance table
df_balance_out = df_balance.reset_index().rename(columns={"index": "date"}) if "date" not in df_balance.columns else df_balance
df_balance_out.to_csv(fig_dir / "mass_balance.csv", index=False)

# Write a small metadata file so we know which geometry produced this run
with open(fig_dir / "geometry_info.txt", "w") as fh:
    fh.write(f"GEOM = {GEOM}\n")
    fh.write(f"delta_area_km2 = {delta_area_m2 / 1e6:.1f}\n")
    if GEOM == "okavango_bbox":
        fh.write(f"bbox = {okavango_bbox}\n")
    elif GEOM == "poly_bbox":
        fh.write(f"bbox = {poly_bbox}\n")
    else:
        fh.write(f"shapefile = {delta_shp.name}\n")

print(f"Saved CSVs and geometry info to {fig_dir.resolve()}")


In [ ]:
import matplotlib.pyplot as plt

# --- Hydrology-themed colors + line styles for mass-balance terms ---
TERM_STYLE = {
    "Qin (Mohembo)": {"color": "#0570b0", "ls": "-",    "lw": 1.2},  # strong blue, solid
    "P (CHIRPS)":    {"color": "#31a354", "ls": "--",   "lw": 1.2},  # forest green, dashed
    "ET (median)":   {"color": "#8856a7", "ls": "-.",   "lw": 1.2},  # muted purple, dash-dot
    "ΔS (GRACE)":    {"color": "#c51b7d", "ls": ":",    "lw": 1.4},  # raspberry, dotted
}
CUM_STYLE = {
    "∑ΔS (GRACE)":   {"color": "#c51b7d", "ls": ":",  "lw": 1.8},  # raspberry, dotted
    "∑(Qin+P-ET)":   {"color": "#0570b0", "ls": "-",  "lw": 1.8},  # strong blue, solid
}

# --- use only months where closure is possible ---
dfc = df_balance[df_balance["has_closure"]].copy()

# Save median-ET plots in their own subfolder
median_dir = fig_dir / "median_ET"
median_dir.mkdir(parents=True, exist_ok=True)

# -----------------------
# Plot 1: Terms in km³/month (bar-style)
# dS = Qin + P - ET - (Qout+G)
# -----------------------
terms = dfc[["Qin_km3", "Pkm3", "ETkm3", "dS_km3", "Qout_plus_G_km3"]].copy()
# Make all "inflows" positive and all "outflows" negative for easy visual sum
plot_terms = pd.DataFrame(index=terms.index)
plot_terms["Qin (Mohembo)"] = terms["Qin_km3"]
plot_terms["P (CHIRPS)"]    = terms["Pkm3"]
plot_terms["ET (median)"]   = - terms["ETkm3"]
plot_terms["ΔS (GRACE)"]    = terms["dS_km3"]          # can be ±
# plot_terms["-(Qout+G)"]     = -terms["Qout_plus_G_km3"]  # shown as an outflow

fig, ax = plt.subplots(figsize=(12, 5))
for col_name in plot_terms.columns:
    sty = TERM_STYLE.get(col_name, {"color": "0.5", "ls": "-", "lw": 1})
    plot_terms[col_name].rolling(3, center=True).mean().plot(
        ax=ax, color=sty["color"], linestyle=sty["ls"], linewidth=sty["lw"], label=col_name
    )
ax.axhline(0, lw=1, c="0.4")
ax.set_title(f"Okavango Delta monthly mass balance terms (3-mo smoothed) [{GEOM}]")
ax.set_ylabel("km³ / month")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
fig.savefig(median_dir / "mass_balance_terms.png", dpi=150, bbox_inches="tight")
plt.show()


# -----------------------
# Plot 3: Cumulative anomalies (integrate monthly terms; km³)
# -----------------------
cum = pd.DataFrame(index=dfc.index)
cum["∑ΔS (GRACE)"] = dfc["dS_km3"].cumsum()
cum["∑(Qin+P-ET)"] = (dfc["Qin_km3"] + dfc["Pkm3"] - dfc["ETkm3"]).cumsum()
# cum["∑(Qout+G)"]   = dfc["Qout_plus_G_km3"].cumsum()

fig, ax = plt.subplots(figsize=(12, 4))
for col_name in cum.columns:
    sty = CUM_STYLE.get(col_name, {"color": "0.5", "ls": "-", "lw": 1.5})
    cum[col_name].plot(ax=ax, color=sty["color"], linestyle=sty["ls"], linewidth=sty["lw"], label=col_name)
ax.axhline(0, lw=1, c="0.4")
ax.set_title(f"Cumulative sums [{GEOM}]")
ax.set_ylabel("km³ (cumulative)")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
fig.savefig(median_dir / "cumulative_sums.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
et_km3_wide

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- months where closure is possible for non-ET terms (Qin, P, dS must exist) ---
base_mask = df_balance[["Qin_km3", "Pkm3", "dS_km3"]].notna().all(axis=1)
dfc_base = df_balance.loc[base_mask].copy()

# --- identify ET ensemble columns (wide table assumed to exist) ---
# Preferred: use the wide ET table you already created earlier (et_km3_wide)
# If it doesn't exist, rebuild from df_all:
try:
    et_km3_wide
except NameError:
    et_km3_wide = (df_all.pivot_table(index="date", columns="dataset", values="et_km3_total", aggfunc="mean")
                        .sort_index())

# Align ET columns to df_balance index and keep only those with enough coverage
et_km3_wide = et_km3_wide.reindex(dfc_base.index)

min_frac = 0.2  # require >=70% non-NaN in the plotted window
keep = (et_km3_wide.notna().mean() >= min_frac)
et_cols = et_km3_wide.columns[keep].tolist()


print(f"Plotting {len(et_cols)} ET products (coverage >= {int(min_frac*100)}%).")

for et_name in et_cols:
    dfc = dfc_base.copy()
    dfc["ETkm3_model"] = et_km3_wide[et_name]

    # mask to months where THIS ET product exists
    dfc = dfc[dfc["ETkm3_model"].notna()].copy()

    et_slug = et_name.replace("/", "_").replace(" ", "_")
    model_dir = fig_dir / "single-model" / et_slug
    model_dir.mkdir(parents=True, exist_ok=True)

    # --- Plot 1: monthly terms (3-mo smooth) ---
    et_label = f"ET ({et_name})"
    term_style = {
        "Qin (Mohembo)": {"color": "#0570b0", "ls": "-",  "lw": 1.2},
        "P (CHIRPS)":    {"color": "#31a354", "ls": "--", "lw": 1.2},
        et_label:        {"color": "#8856a7", "ls": "-.", "lw": 1.2},
        "ΔS (GRACE)":    {"color": "#c51b7d", "ls": ":",  "lw": 1.4},
    }

    plot_terms = pd.DataFrame(index=dfc.index)
    plot_terms["Qin (Mohembo)"] = dfc["Qin_km3"]
    plot_terms["P (CHIRPS)"]    = dfc["Pkm3"]
    plot_terms[et_label] = -dfc["ETkm3_model"]
    plot_terms["ΔS (GRACE)"]    = dfc["dS_km3"]

    fig, ax = plt.subplots(figsize=(12, 4.5))
    for col_name in plot_terms.columns:
        sty = term_style.get(col_name, {"color": "0.5", "ls": "-", "lw": 1})
        plot_terms[col_name].rolling(3, center=True).mean().plot(
            ax=ax, color=sty["color"], linestyle=sty["ls"], linewidth=sty["lw"], label=col_name
        )
    ax.axhline(0, lw=1, c="0.4")
    ax.set_title(f"Mass balance terms (3-mo smoothed) using ET = {et_name}")
    ax.set_ylabel("km³ / month")
    ax.set_xlabel("")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.tight_layout()
    fig.savefig(model_dir / "mass_balance_terms.png", dpi=150, bbox_inches="tight")
    plt.show()

    # --- Plot 2: cumulative comparison ---
    cum_label = f"∑(Qin+P-ET)  [ET={et_name}]"
    cum_style = {
        "∑ΔS (GRACE)": {"color": "#c51b7d", "ls": ":", "lw": 1.8},
        cum_label:      {"color": "#0570b0", "ls": "-", "lw": 1.8},
    }

    cum = pd.DataFrame(index=dfc.index)
    cum["∑ΔS (GRACE)"] = dfc["dS_km3"].cumsum()
    cum[cum_label] = (dfc["Qin_km3"] + dfc["Pkm3"] - dfc["ETkm3_model"]).cumsum()

    fig, ax = plt.subplots(figsize=(12, 3.8))
    for col_name in cum.columns:
        sty = cum_style.get(col_name, {"color": "0.5", "ls": "-", "lw": 1.5})
        cum[col_name].plot(ax=ax, color=sty["color"], linestyle=sty["ls"], linewidth=sty["lw"], label=col_name)
    ax.axhline(0, lw=1, c="0.4")
    ax.set_title(f"Cumulative sums using ET = {et_name}")
    ax.set_ylabel("km³ (cumulative)")
    ax.set_xlabel("")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.tight_layout()
    fig.savefig(model_dir / "cumulative_sums.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# ============================================================
# Cross-geometry comparison
# Load mass_balance.csv from each geometry folder and compare
# key water-balance terms side-by-side.
# NOTE: ETmm and Pmm are area-normalised and SHOULD differ by geometry.
#       Qin_m3s is a fixed gauge measurement and should be IDENTICAL across
#       geometries — use it (not Qin_mm) to verify consistency.
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path("../figures/ET comparison")
GEOMS = ["delta_polygon", "poly_bbox", "okavango_bbox"]
COLORS = {"delta_polygon": "#8c2d04", "poly_bbox": "#fdae6b", "okavango_bbox": "#d95f02"}

dfs_cmp = {}
for g in GEOMS:
    fp = BASE / g / "mass_balance.csv"
    if fp.exists():
        df_ = pd.read_csv(fp, parse_dates=["date"])
        df_["geom"] = g
        dfs_cmp[g] = df_
    else:
        print(f"  [missing] {fp}")

if not dfs_cmp:
    print("No geometry outputs found yet — run and save each geometry first.")
else:
    # --- Plot: ET, P (area-normalised, vary by geometry) + raw discharge (geometry-invariant) ---
    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

    for ax, col, label in zip(
        axes,
        ["ETmm", "Pmm", "Qin_m3s"],
        ["ET (mm/month)", "CHIRPS P (mm/month)", "Mohembo Qin (m³/s)  ← should overlap"],
    ):
        for g, df_ in dfs_cmp.items():
            if col in df_.columns:
                ax.plot(df_["date"], df_[col], label=g, color=COLORS[g], linewidth=1)
        ax.set_ylabel(label)
        ax.axhline(0, lw=0.8, ls=":", c="0.5")
        ax.legend(loc="upper right", fontsize=8)

    axes[0].set_title("Water-balance terms by aggregation geometry")
    axes[-1].set_xlabel("Date")
    plt.tight_layout()
    fig.savefig(BASE / "comparison_ET_P_Q.png", dpi=150, bbox_inches="tight")
    plt.show()

    # --- Summary table: mean annual values ---
    rows = []
    for g, df_ in dfs_cmp.items():
        info_fp = BASE / g / "geometry_info.txt"
        area_km2 = None
        if info_fp.exists():
            for line in info_fp.read_text().splitlines():
                if "delta_area_km2" in line:
                    area_km2 = float(line.split("=")[1].strip())
        rows.append({
            "geometry": g,
            "area_km2": area_km2,
            "mean_ETmm_yr": df_["ETmm"].mean() * 12 if "ETmm" in df_.columns else None,
            "mean_Pmm_yr": df_["Pmm"].mean() * 12 if "Pmm" in df_.columns else None,
            # Qin in m³/s — should be the same for all geometries
            "mean_Qin_m3s": df_["Qin_m3s"].mean() if "Qin_m3s" in df_.columns else None,
        })

    summary = pd.DataFrame(rows).set_index("geometry")
    print("\nAnnual mean summary across geometries:")
    display(summary.round(1))
    summary.to_csv(BASE / "geometry_comparison_summary.csv")
    print(f"Saved: {(BASE / 'geometry_comparison_summary.csv').resolve()}")


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- Comparative cumulative mass balance by polygon region (ensemble-mean ET) ---
fig, ax = plt.subplots(figsize=(13, 5))

for g, df_ in dfs_cmp.items():
    # Only months where closure is possible
    mask = df_[["Qin_km3", "Pkm3", "ETkm3", "dS_km3"]].notna().all(axis=1)
    dfc = df_.loc[mask].copy().sort_values("date").set_index("date")

    # Cumulative observed storage change (GRACE)
    cum_dS = dfc["dS_km3"].cumsum()
    ax.plot(cum_dS.index, cum_dS.values, color=COLORS[g], lw=2, ls="--", alpha=0.6,
            label=f"∑ΔS GRACE [{g}]")

    # Cumulative (Qin + P − ET) using ensemble-mean ET
    cum_qpe = (dfc["Qin_km3"] + dfc["Pkm3"] - dfc["ETkm3"]).cumsum()
    ax.plot(cum_qpe.index, cum_qpe.values, color=COLORS[g], lw=1.5,
            label=f"∑(Qin+P−ET) [{g}]")

ax.axhline(0, lw=0.8, ls=":", c="0.5")
ax.set_title("Comparative cumulative mass balance by polygon region (ensemble-mean ET)")
ax.set_ylabel("km³ (cumulative)")
ax.set_xlabel("")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=12)
plt.tight_layout()
fig.savefig(BASE / "comparative_cumulative_sums.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# --- Comparative cumulative mass balance by polygon region (ensemble-MEDIAN ET) ---
fig, ax = plt.subplots(figsize=(13, 5))

for g, df_mb in dfs_cmp.items():
    # Load per-product ET and compute median across products each month
    et_fp = BASE / g / "et_monthly.csv"
    if not et_fp.exists():
        print(f"  [missing ET] {et_fp}")
        continue
    df_et_long = pd.read_csv(et_fp, parse_dates=["date"])
    et_median_km3 = (df_et_long.pivot_table(index="date", columns="dataset",
                                             values="et_km3_total", aggfunc="mean")
                     .median(axis=1, skipna=True)
                     .rename("ETkm3_median"))

    # Merge median ET into mass-balance table
    dfc = df_mb.copy().sort_values("date").set_index("date")
    dfc = dfc.join(et_median_km3, how="left")

    # Closure mask using median ET
    mask = dfc[["Qin_km3", "Pkm3", "ETkm3_median", "dS_km3"]].notna().all(axis=1)
    dfc = dfc.loc[mask]

    # Cumulative observed storage change (GRACE)
    cum_dS = dfc["dS_km3"].cumsum()
    ax.plot(cum_dS.index, cum_dS.values, color=COLORS[g], lw=2, ls="--", alpha=0.6,
            label=f"∑ΔS GRACE [{g}]")

    # Cumulative (Qin + P − ET_median)
    cum_qpe = (dfc["Qin_km3"] + dfc["Pkm3"] - dfc["ETkm3_median"]).cumsum()
    ax.plot(cum_qpe.index, cum_qpe.values, color=COLORS[g], lw=1.5,
            label=f"∑(Qin+P−ET) [{g}]")

ax.axhline(0, lw=0.8, ls=":", c="0.5")
ax.set_title("Comparative cumulative mass balance by polygon region (ensemble-median ET)")
ax.set_ylabel("km³ (cumulative)")
ax.set_xlabel("")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=12)
plt.tight_layout()
fig.savefig(BASE / "comparative_cumulative_sums_median.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# --- Load per-product ET for each geometry and find the common set of ET models ---
et_wide_by_geom = {}
for g in dfs_cmp:
    et_fp = BASE / g / "et_monthly.csv"
    if not et_fp.exists():
        print(f"  [missing ET] {et_fp}")
        continue
    df_et_long = pd.read_csv(et_fp, parse_dates=["date"])
    et_wide = df_et_long.pivot_table(index="date", columns="dataset",
                                      values="et_km3_total", aggfunc="mean")
    et_wide_by_geom[g] = et_wide

# Union of all ET product names across geometries
all_et_names = sorted(set().union(*(w.columns for w in et_wide_by_geom.values())))

# --- One plot per ET product: comparative cumulative sums across polygon regions ---
for et_name in all_et_names:
    fig, ax = plt.subplots(figsize=(13, 5))
    any_plotted = False

    for g, df_mb in dfs_cmp.items():
        et_wide = et_wide_by_geom.get(g)
        if et_wide is None or et_name not in et_wide.columns:
            continue

        dfc = df_mb.copy().sort_values("date").set_index("date")
        dfc["ETkm3_model"] = et_wide[et_name]

        mask = dfc[["Qin_km3", "Pkm3", "ETkm3_model", "dS_km3"]].notna().all(axis=1)
        dfc = dfc.loc[mask]
        if len(dfc) < 6:
            continue

        # Cumulative GRACE ΔS
        cum_dS = dfc["dS_km3"].cumsum()
        ax.plot(cum_dS.index, cum_dS.values, color=COLORS[g], lw=2, ls="--", alpha=0.6,
                label=f"∑ΔS GRACE [{g}]")

        # Cumulative (Qin + P − ET)
        cum_qpe = (dfc["Qin_km3"] + dfc["Pkm3"] - dfc["ETkm3_model"]).cumsum()
        ax.plot(cum_qpe.index, cum_qpe.values, color=COLORS[g], lw=1.5,
                label=f"∑(Qin+P−ET) [{g}]")
        any_plotted = True

    if not any_plotted:
        plt.close(fig)
        continue

    ax.axhline(0, lw=0.8, ls=":", c="0.5")
    ax.set_title(f"Cumulative mass balance by region — ET = {et_name}")
    ax.set_ylabel("km³ (cumulative)")
    ax.set_xlabel("")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=9)
    plt.tight_layout()
    slug = et_name.replace("/", "_").replace(" ", "_")
    fig.savefig(BASE / f"cumulative_by_region_{slug}.png", dpi=150, bbox_inches="tight")
    plt.show()


Last edited: 2026-03-06
